# Day 6 — Course Recommender TF-IDF Pipeline
## Notebook: `04_courses_pipeline.ipynb`

**Goal:** Build the sklearn Pipeline that turns the Coursera CSV into 3 trained  
pickles ready for the FastAPI backend.

**Algorithm:** TF-IDF vectorisation + cosine similarity  
(content-based filtering — no user ratings needed)

**Pipeline stages:**  
`CourseLoader` → `CourseCleaner` → `TFIDFTransformer` → `CourseSimilarityBuilder`

**Output pickles** (saved to `../models/`):  
- `course_tfidf.pkl`  — fitted TfidfVectorizer  
- `course_sim.pkl`    — cosine similarity matrix  
- `courses_df.pkl`    — full course DataFrame with combined_text  

> Run all cells top to bottom. `../data/coursera_courses.csv` must exist.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import warnings
import os
warnings.filterwarnings('ignore')

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

os.makedirs('../models', exist_ok=True)
print("Imports successful")

## 1. Custom transformer: CourseLoader

In [ ]:
class CourseLoader(BaseEstimator, TransformerMixin):
    """
    Stage 1: Load the Coursera CSV and standardise column names.

    Different Kaggle versions of the dataset use different column names.
    This transformer detects whichever naming convention is present
    and maps everything to a standard set:
      course_title, subject, level, rating, course_url

    Input  : None
    Output : dict with key 'raw_df'
    """
    def __init__(self, csv_path='../data/coursera_courses.csv'):
        self.csv_path = csv_path

    def fit(self, X=None, y=None):
        return self

    def transform(self, X=None):
        df = pd.read_csv(self.csv_path)
        print(f"[CourseLoader]  loaded {df.shape[0]:,} rows  x  {df.shape[1]} columns")
        print(f"[CourseLoader]  original columns: {df.columns.tolist()}")

        col_map = {}

        # Detect title column
        for c in ['Course Name', 'course_title', 'name', 'title', 'Course Title', 'CourseName']:
            if c in df.columns:
                col_map[c] = 'course_title'
                break

        # Detect subject / category
        for c in ['Course Description', 'subject', 'category', 'Subject',
                  'skills', 'Category', 'Description']:
            if c in df.columns:
                col_map[c] = 'subject'
                break

        # Detect level / difficulty
        for c in ['Difficulty Level', 'level', 'difficulty', 'Level', 'DifficultyLevel']:
            if c in df.columns:
                col_map[c] = 'level'
                break

        # Detect rating
        for c in ['Course Rating', 'rating', 'Rating', 'score', 'CourseRating']:
            if c in df.columns:
                col_map[c] = 'rating'
                break

        # Detect URL
        for c in ['Course URL', 'course_url', 'url', 'URL', 'link', 'CourseURL']:
            if c in df.columns:
                col_map[c] = 'course_url'
                break

        df = df.rename(columns=col_map)
        return {'raw_df': df}

print("CourseLoader defined")

## 2. Custom transformer: CourseCleaner

In [ ]:
class CourseCleaner(BaseEstimator, TransformerMixin):
    """
    Stage 2 — IMPROVED v2
    =====================
    Key change: combined_text now weights fields by repetition:
      - course_title repeated 3x  (strongest relevance signal)
      - subject       repeated 2x  (important category signal)
      - level                 1x
      - description    up to 1x  (included when available — adds semantic depth)

    Why repetition instead of explicit TF-IDF weights?
    TfidfVectorizer has no per-field weight parameter. Repeating a field
    N times multiplies its term frequency by N before IDF normalisation —
    the standard IR technique for field weighting without custom vectorisers.
    """
    LEVEL_MAP = {
        'beginner': 'Beginner', 'intermediate': 'Intermediate',
        'advanced': 'Advanced', 'mixed': 'Mixed', 'all levels': 'Mixed',
        'all': 'Mixed', 'expert': 'Advanced',
    }

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        import re
        data = X.copy()
        df   = data['raw_df'].copy()

        # Keep columns we need
        keep = [c for c in ['course_title','subject','level','rating','course_url','description']
                if c in df.columns]
        df = df[keep].copy()

        for col, default in [('subject','General'),('level','Mixed'),
                              ('rating',4.0),('course_url',''),('description','')]:
            if col not in df.columns:
                df[col] = default

        df.dropna(subset=['course_title'], inplace=True)
        df['subject']     = df['subject'].fillna('General')
        df['description'] = df['description'].fillna('').astype(str)
        df['course_url']  = df['course_url'].fillna('')

        # Normalise rating
        df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
        df['rating'] = df['rating'].fillna(df['rating'].median()).clip(0, 5)

        # Standardise level
        df['level'] = (df['level'].astype(str).str.strip().str.lower()
                        .map(self.LEVEL_MAP).fillna('Mixed'))

        df.drop_duplicates(subset=['course_title'], keep='first', inplace=True)
        df.reset_index(drop=True, inplace=True)

        # ── Build weighted combined_text (v2 improvement) ──────────────
        def clean(s): return re.sub(r'[^a-z0-9 ]', ' ', str(s).lower().strip())

        def make_text(row):
            title = clean(row.get('course_title', ''))
            subj  = clean(row.get('subject',      ''))
            level = clean(row.get('level',        ''))
            desc  = clean(str(row.get('description', ''))[:200])
            # title x3, subject x2, level x1, description x1
            return ' '.join(filter(None, [
                title, title, title,
                subj,  subj,
                level,
                desc,
            ]))

        df['combined_text'] = df.apply(make_text, axis=1)

        data['courses_df'] = df
        print(f"[CourseCleaner v2] {df.shape}")
        print(f"[CourseCleaner v2] levels: {sorted(df['level'].unique())}")
        print(f"[CourseCleaner v2] sample combined_text: '{df['combined_text'].iloc[0][:80]}...'")
        return data

print("CourseCleaner v2 defined — title x3, subject x2, description included")


## 3. Custom transformer: TFIDFTransformer

In [ ]:
class TFIDFTransformer(BaseEstimator, TransformerMixin):
    """
    Stage 3 — IMPROVED v2
    =====================
    Changes vs v1:
      max_features : 5,000  → 10,000   captures specialist terms
      ngram_range  : (1,2)  → (1,3)    trigrams: 'machine learning python'
      min_df       : 1      → 2        removes single-occurrence noise words
      max_df       : 1.0    → 0.85     removes near-universal words ('course','learn')
      sublinear_tf : True   → True     unchanged — log-scales TF
    """
    def __init__(self, max_features=10000, ngram_range=(1, 3)):
        self.max_features = max_features
        self.ngram_range  = ngram_range

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        data = X.copy()
        df   = data['courses_df']

        vectorizer = TfidfVectorizer(
            max_features = self.max_features,
            ngram_range  = self.ngram_range,
            sublinear_tf = True,
            min_df       = 2,      # remove hapax-legomena (noise)
            max_df       = 0.85,   # remove near-universal terms
            analyzer     = 'word',
            token_pattern= r'\b[a-z][a-z0-9]{1,}\b',
        )
        tf_matrix = vectorizer.fit_transform(df['combined_text'])

        print(f"[TFIDFTransformer v2] vocab={len(vectorizer.vocabulary_):,}  "
              f"tf_matrix={tf_matrix.shape}")
        print(f"[TFIDFTransformer v2] ngram_range={self.ngram_range}  "
              f"max_features={self.max_features}")

        data['vectorizer'] = vectorizer
        data['tf_matrix']  = tf_matrix
        return data

print("TFIDFTransformer v2 defined — max_features=10000, ngram=(1,3), min_df=2, max_df=0.85")


## 4. Custom transformer: CourseSimilarityBuilder

In [ ]:
class CourseSimilarityBuilder(BaseEstimator, TransformerMixin):
    """
    Stage 4: Pre-compute the full cosine similarity matrix.

    Shape: (n_courses, n_courses)
    sim_matrix[i, j] = cosine similarity between course i and course j

    Pre-computing is a memory-for-speed tradeoff:
      - Pro: near-instant query responses (just index a row)
      - Con: ~50MB storage for 3500 courses
      - Alternative: compute at query time (~30ms overhead per request)

    For a dataset of <5000 courses, pre-computing is the right choice.

    Input  : data dict with 'tf_matrix'
    Output : data dict with 'sim_matrix' added
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        data      = X.copy()
        tf_matrix = data['tf_matrix']

        # Dense cosine similarity matrix
        sim_matrix = cosine_similarity(tf_matrix, tf_matrix)

        data['sim_matrix'] = sim_matrix
        print(f"[CourseSimilarityBuilder]  similarity matrix: {sim_matrix.shape}")
        print(f"[CourseSimilarityBuilder]  matrix size: "
              f"{sim_matrix.nbytes / (1024**2):.1f} MB in memory")
        return data

print("CourseSimilarityBuilder defined")

## 5. Assemble and run the pipeline

In [ ]:
# ── IMPROVED COURSE PIPELINE v2 ───────────────────────────────────
course_pipeline = Pipeline(steps=[
    ('loader',      CourseLoader()),
    ('cleaner',     CourseCleaner()),          # v2: title x3, subject x2, description included
    ('tfidf',       TFIDFTransformer(
                        max_features=10000,    # was 5000 — captures more specialist terms
                        ngram_range=(1, 3)     # was (1,2) — adds trigrams
                    )),
    ('similarity',  CourseSimilarityBuilder()),
])

print("Course pipeline v2 stages:")
for name, step in course_pipeline.steps:
    print(f"  {name:12s} → {step.__class__.__name__}")
print()

print("Running improved pipeline v2...")
output = course_pipeline.fit_transform(None)
print()
print("Pipeline complete.")
print(f"Courses in model : {len(output['courses_df']):,}")
print(f"Vocab size       : {len(output['vectorizer'].vocabulary_):,}")
print(f"Similarity matrix: {output['sim_matrix'].shape}")


## 6. Recommendation function — test before pickling

In [ ]:
def recommend_courses(topic: str, level: str = None,
                      branch: str = None, n: int = 10,
                      courses_df=None, vectorizer=None, tf_matrix=None):
    """
    Return top-n course recommendations for a topic query.

    Parameters
    ----------
    topic      : str   query string (e.g. 'machine learning', 'python basics')
    level      : str   optional filter — 'Beginner', 'Intermediate', 'Advanced'
    branch     : str   optional — 'CSE' applies 1.2x boost to CS/ML courses
    n          : int   number of results (default 10)
    
    Returns
    -------
    list of dicts with keys: course_title, subject, level, rating, course_url, score
    """
    if courses_df  is None: courses_df  = output['courses_df']
    if vectorizer  is None: vectorizer  = output['vectorizer']
    if tf_matrix   is None: tf_matrix   = output['tf_matrix']

    # Transform query using fitted vectorizer
    query_vec = vectorizer.transform([topic.lower()])

    # Compute cosine similarity between query and all courses
    scores = cosine_similarity(query_vec, tf_matrix).flatten()

    result_df = courses_df.copy()
    result_df['score'] = scores

    # Apply level filter if specified
    if level:
        result_df = result_df[
            result_df['level'].str.lower() == level.lower()
        ]
        if len(result_df) == 0:
            # Fall back to all levels if filter leaves nothing
            result_df = courses_df.copy()
            result_df['score'] = scores

    # Apply branch boost (CSE users get CS/ML/Data courses ranked higher)
    if branch and branch.upper() == 'CSE':
        cs_keywords = r'computer|data|machine|python|programming|software|algorithm|network'
        cs_mask = result_df['subject'].str.contains(cs_keywords, case=False, na=False)
        result_df.loc[cs_mask, 'score'] *= 1.2

    # Sort by score and take top-n
    top = result_df.nlargest(n, 'score')

    results = []
    for _, row in top.iterrows():
        results.append({
            'course_title': row['course_title'],
            'subject'     : row['subject'],
            'level'       : row['level'],
            'rating'      : round(float(row['rating']), 2),
            'course_url'  : row['course_url'],
            'score'       : round(float(row['score']), 4),
        })
    return results


# ── Test with 5 different queries ──────────────────────────────────
test_queries = [
    ('machine learning', None, None),
    ('python programming', 'Beginner', None),
    ('data science', None, 'CSE'),
    ('web development', 'Intermediate', None),
    ('algorithms', None, 'CSE'),
]

for topic, level, branch in test_queries:
    results = recommend_courses(topic, level=level, branch=branch, n=5)
    print(f"\nQuery: '{topic}'  level={level}  branch={branch}")
    for i, r in enumerate(results[:3], 1):
        print(f"  {i}. [{r['score']:.3f}] {r['course_title'][:55]}  ({r['level']})")

## 7. Hybrid boost verification

In [ ]:
# Show that CSE branch boost changes the ranking for CS-related queries
print("=== Branch boost effect for 'data science' query ===")
print()

results_no_boost = recommend_courses('data science', branch=None, n=5)
results_boosted  = recommend_courses('data science', branch='CSE', n=5)

print("Without branch boost:")
for i, r in enumerate(results_no_boost[:5], 1):
    print(f"  {i}. [{r['score']:.3f}]  {r['course_title'][:50]}")

print("\nWith CSE branch boost:")
for i, r in enumerate(results_boosted[:5], 1):
    print(f"  {i}. [{r['score']:.3f}]  {r['course_title'][:50]}")

## 8. Export 3 pickles to models/

In [ ]:
artifacts = {
    '../models/course_tfidf.pkl' : output['vectorizer'],
    '../models/course_sim.pkl'   : output['sim_matrix'],
    '../models/courses_df.pkl'   : output['courses_df'],
}

for path, obj in artifacts.items():
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"  Saved: {path:42s}  ({size_mb:.1f} MB)")

print("\nAll 3 course pickles saved to ../models/")

## 9. Verify pickles round-trip correctly

In [ ]:
with open('../models/course_tfidf.pkl', 'rb') as f: chk_vec = pickle.load(f)
with open('../models/course_sim.pkl',   'rb') as f: chk_sim = pickle.load(f)
with open('../models/courses_df.pkl',   'rb') as f: chk_df  = pickle.load(f)

print("Pickle verification:")
print(f"  course_tfidf.pkl : {type(chk_vec).__name__}  vocab={len(chk_vec.vocabulary_):,}")
print(f"  course_sim.pkl   : ndarray {chk_sim.shape}")
print(f"  courses_df.pkl   : DataFrame {chk_df.shape}")
print(f"  columns          : {chk_df.columns.tolist()}")

# End-to-end test from pickled vectorizer (not precomputed sim matrix)
tf_from_pickle = chk_vec.transform(chk_df['combined_text'])
test_query     = chk_vec.transform(['machine learning python'])
scores         = cosine_similarity(test_query, tf_from_pickle).flatten()

chk_df_copy = chk_df.copy()
chk_df_copy['score'] = scores
top5 = chk_df_copy.nlargest(5, 'score')

print("\nEnd-to-end test for 'machine learning python':")
for _, row in top5.iterrows():
    print(f"  [{row['score']:.3f}]  {row['course_title'][:55]}")

print("\nAll checks passed — Day 6 complete!")

## 10. Methodology notes for BTP Report

### TF-IDF Content-Based Filtering

**TF-IDF (Term Frequency – Inverse Document Frequency)** converts text into numerical  
vectors where each dimension represents a word (or bigram).

**TF (Term Frequency):** how often a word appears in a course's combined_text.  
With `sublinear_tf=True`, we apply log(1 + tf) to reduce the dominance of very common words.

**IDF (Inverse Document Frequency):** log(N / df) where N = total courses, df = courses containing the word.  
Words that appear in every course ('the', 'and', 'course') get very low IDF — they don't help distinguish courses.  
Rare meaningful words ('bayesian', 'kubernetes', 'reinforcement') get high IDF.

**Cosine similarity** measures the angle between two TF-IDF vectors.  
A similarity of 1.0 means identical topics; 0.0 means completely unrelated.  
Unlike dot product, cosine similarity is length-independent — a 5-word title  
and a 20-word description can still score 1.0 if they discuss the same topic.

**Bigrams** (`ngram_range=(1,2)`) capture two-word phrases like 'machine learning',  
'data science', 'neural network' as single features — much more informative than  
treating each word independently.

**Why TF-IDF over collaborative filtering for courses?**  
The Coursera dataset has no user-course rating matrix — we don't know which users  
took which courses. TF-IDF requires only the course metadata (title, subject, level),  
making it ideal for content-based recommendations without user data.